In [4]:
!pip install datasets

  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)


You should consider upgrading via the 'C:\Users\vi120\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
import os, pandas as pd
from PIL import Image
from datasets import load_dataset, Image as HFImage
from huggingface_hub import login

In [ ]:
login(token=" ") #your token hf

ds = load_dataset("5CD-AI/Viet-Doc-VQA", split="train")
ds_all = load_dataset("5CD-AI/Viet-Doc-VQA")

Generating train split: 100%|██████████| 51856/51856 [00:25<00:00, 2060.38 examples/s]


ValueError: Unknown split "test". Should be one of ['train'].

In [10]:
ds.save_to_disk("Viet-Doc-VQA_arrow") 

Saving the dataset (11/11 shards): 100%|██████████| 51856/51856 [00:54<00:00, 946.68 examples/s]


In [ ]:
ds = ds.cast_column("image", HFImage())

os.makedirs("Viet-Doc-VQA/images", exist_ok=True)
rows = []

for i, ex in enumerate(ds):
    img_obj = ex["image"]
    # Trường hợp có path trong cache
    if isinstance(img_obj, dict) and "path" in img_obj and img_obj["path"] is not None:
        pil = Image.open(img_obj["path"]).convert("RGB")
    else:
        # Fallback: img_obj có dạng PIL hoặc bytes
        if isinstance(img_obj, Image.Image):
            pil = img_obj.convert("RGB")
        else:
            # khi img_obj là {"bytes": ...}
            pil = Image.open(io.BytesIO(img_obj["bytes"])).convert("RGB")

    out_path = f"Viet-Doc-VQA/images/{i:06d}.jpg"
    pil.save(out_path, quality=95)

    rows.append({
        "id": ex.get("id", i),
        "image_path": out_path,
        "description": ex.get("description", ""),
        "conversations": ex.get("conversations", "")
    })

pd.DataFrame(rows).to_csv("Viet-Doc-VQA/metadata.csv", index=False)
print("Đã xuất ảnh vào Viet-Doc-VQA/images và metadata.csv")


Đã xuất ảnh vào Viet-Doc-VQA/images và metadata.csv
